## Import libraries and configure the environment

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

PROJECT_ROOT = Path.cwd().parent
# Reproducibility
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Select computation device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

PyTorch version: 2.15.0+cpu
CUDA available: False
Selected device: cpu


## Understand the input data

Our Human Activity Recognition dataset contains six sensor channels
recorded over 128 time steps for each sample.

The CNN expects input in the following format:

(batch_size, channels, sequence_length)

For a batch size of 64, the expected shape is:

(64, 6, 128)

In [6]:

# Create a sample batch to understand the input dimensions
dummy_input = torch.randn(64, 6, 128)

print("Input shape:", dummy_input.shape)
print("Batch size:", dummy_input.shape[0])
print("Number of sensor channels:", dummy_input.shape[1])
print("Time steps per sample:", dummy_input.shape[2])


Input shape: torch.Size([64, 6, 128])
Batch size: 64
Number of sensor channels: 6
Time steps per sample: 128


## Load prepared datasets

Load the preprocessed sensor signals, activity labels, and
participant-independent training and validation split.

In [8]:

import sys
from torch.utils.data import Dataset, DataLoader, Subset

# Make the project root available for importing paths.py
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import centralized project paths
from paths import PROJECT_ROOT, DATA_DIR

# Define the processed data file
PROCESSED_DATA_FILE = DATA_DIR / "processed" / "har_preprocessed.pt"

# Load saved data
data = torch.load(
    PROCESSED_DATA_FILE,
    map_location=device,
    weights_only=False
)

# Retrieve tensors and split indices
X_train_tensor = data["X_train"]
X_test_tensor = data["X_test"]

y_train_tensor = data["y_train"]
y_test_tensor = data["y_test"]

train_indices = data["train_indices"]
val_indices = data["val_indices"]

print("Data loaded successfully!")
print("Training signals:", X_train_tensor.shape)
print("Training labels:", y_train_tensor.shape)
print("Testing signals:", X_test_tensor.shape)
print("Testing labels:", y_test_tensor.shape)
print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))


Data loaded successfully!
Training signals: torch.Size([7352, 6, 128])
Training labels: torch.Size([7352])
Testing signals: torch.Size([2947, 6, 128])
Testing labels: torch.Size([2947])
Training indices: 5551
Validation indices: 1801


## Create PyTorch datasets

Create a custom Dataset to organize sensor signals and their
corresponding activity labels.

In [9]:

class HAR_Dataset(Dataset):

    def __init__(self, signals, labels):
        self.signals = signals
        self.labels = labels

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]


# Create the complete training and testing datasets
train_dataset = HAR_Dataset(X_train_tensor, y_train_tensor)
test_dataset = HAR_Dataset(X_test_tensor, y_test_tensor)

print("Training dataset size:", len(train_dataset))
print("Testing dataset size:", len(test_dataset))


Training dataset size: 7352
Testing dataset size: 2947


In [ ]:
# Create training and validation subsets
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

print("Training subset size:", len(train_subset))
print("Validation subset size:", len(val_subset))


Training subset size: 5551
Validation subset size: 1801


In [11]:
# Create DataLoaders

BATCH_SIZE = 64

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


Training batches: 87
Validation batches: 29
Testing batches: 47


In [12]:

# Get one batch from the training DataLoader
signals, labels = next(iter(train_loader))

print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)

print("Signal data type:", signals.dtype)
print("Label data type:", labels.dtype)

print("First sample label:", labels[0].item())
print("First sample signal shape:", signals[0].shape)


Signal batch shape: torch.Size([64, 6, 128])
Label batch shape: torch.Size([64])
Signal data type: torch.float32
Label data type: torch.int64
First sample label: 1
First sample signal shape: torch.Size([6, 128])
